In [2]:
import os
import sys
import json
import cv2
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join("..")))

from src.preprocessing import preprocess_image
from src.segmentation import segment_products
from src.detection import detect_products, crop_products, draw_detections, print_detection_summary
from src.classification import load_model, classify_detections

from src.analytics import (
    count_by_category, compute_percentages, build_summary,
    summary_to_dataframe, format_summary_table, print_summary,
    generate_report, aggregate_summaries,
    save_summary_json, save_summary_csv, save_summary_text,
)
from src.visualization import plot_bar_chart, plot_pie_chart, generate_charts

In [3]:
OUTPUT_DIR = "../outputs"
MODEL_DIR = "../models/classifier"
CHECKOUT_DIR = "../data/checkout_easy"   

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
USE_REAL_PIPELINE = False 

image = None
base_name = "demo_basket"
detections = None
classification_results = None

def try_real_pipeline():
    sample_files = [f for f in os.listdir(CHECKOUT_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not sample_files:
        raise FileNotFoundError(f"No checkout images found in {CHECKOUT_DIR}")

    sample_path = os.path.join(CHECKOUT_DIR, sample_files[0])
    name = os.path.splitext(os.path.basename(sample_path))[0]

    img = preprocess_image(sample_path)
    seg = segment_products(img)
    dets = detect_products(img, seg)
    if not dets:
        raise RuntimeError(f"No products detected in {sample_path}")

    crops = crop_products(img, dets)
    model, label_encoder = load_model(MODEL_DIR)
    results = classify_detections(crops, model, label_encoder)
    return img, name, dets, results


if USE_REAL_PIPELINE:
    image, base_name, detections, classification_results = try_real_pipeline()
    print(f"Using REAL pipeline output for '{base_name}'.")
else:
    try:
        image, base_name, detections, classification_results = try_real_pipeline()
        print(f"Real pipeline available — using it for '{base_name}'.")
    except Exception as e:
        print(f"Real pipeline not ready yet ({e}).")
        print("Falling back to DEMO classification results for this notebook.")
        classification_results = [
            {"det_id": 1, "label": "bottle-like",   "confidence": 0.94},
            {"det_id": 2, "label": "bottle-like",   "confidence": 0.88},
            {"det_id": 3, "label": "box-like",      "confidence": 0.79},
            {"det_id": 4, "label": "canister-like", "confidence": 0.66},
            {"det_id": 5, "label": "bag-like",      "confidence": 0.55},
            {"det_id": 6, "label": "canister-like", "confidence": 0.38},  
        ]

classification_results

Real pipeline not ready yet ([Errno 2] No such file or directory: '../models/classifier\\classifier.pkl').
Falling back to DEMO classification results for this notebook.


[{'det_id': 1, 'label': 'bottle-like', 'confidence': 0.94},
 {'det_id': 2, 'label': 'bottle-like', 'confidence': 0.88},
 {'det_id': 3, 'label': 'box-like', 'confidence': 0.79},
 {'det_id': 4, 'label': 'canister-like', 'confidence': 0.66},
 {'det_id': 5, 'label': 'bag-like', 'confidence': 0.55},
 {'det_id': 6, 'label': 'canister-like', 'confidence': 0.38}]

In [5]:
counts = count_by_category(classification_results)
print("Category counts:", dict(counts))

percentages = compute_percentages(counts)
print("Category percentages:")
for category, pct in percentages.items():
    print(f"  {category:<15} {pct:6.2f}%")

Category counts: {'bottle-like': 2, 'box-like': 1, 'canister-like': 2, 'bag-like': 1}
Category percentages:
  bottle-like      33.33%
  box-like         16.67%
  canister-like    33.33%
  bag-like         16.67%


In [6]:
counts_thresholded = count_by_category(classification_results, confidence_threshold=0.5)
print("Category counts (confidence >= 0.5, else 'Uncertain'):", dict(counts_thresholded))

Category counts (confidence >= 0.5, else 'Uncertain'): {'bottle-like': 2, 'box-like': 1, 'canister-like': 1, 'bag-like': 1, 'Uncertain': 1}


In [7]:
summary = build_summary(classification_results, image_name=base_name)
summary

{'image_name': 'demo_basket',
 'generated_at': '2026-09-17T17:07:22',
 'total_products': 6,
 'category_counts': {'bottle-like': 2,
  'canister-like': 2,
  'bag-like': 1,
  'box-like': 1},
 'category_percentages': {'bottle-like': 33.33,
  'canister-like': 33.33,
  'bag-like': 16.67,
  'box-like': 16.67},
 'products': [{'id': 1, 'label': 'bottle-like', 'confidence': 0.94},
  {'id': 2, 'label': 'bottle-like', 'confidence': 0.88},
  {'id': 3, 'label': 'box-like', 'confidence': 0.79},
  {'id': 4, 'label': 'canister-like', 'confidence': 0.66},
  {'id': 5, 'label': 'bag-like', 'confidence': 0.55},
  {'id': 6, 'label': 'canister-like', 'confidence': 0.38}]}

In [8]:
summary_to_dataframe(summary)

,category,count,percentage
0,bottle-like,2,33.33
1,canister-like,2,33.33
2,bag-like,1,16.67
3,box-like,1,16.67
4,TOTAL,6,100.00


In [9]:
print_summary(summary)  

         PRODEXA ANALYSIS
Image: demo_basket
Generated: 2026-09-17T17:07:22

Total Products Detected: 6

Category                 Count         %
----------------------------------------
bottle-like                  2    33.33%
canister-like                2    33.33%
bag-like                     1    16.67%
box-like                     1    16.67%
----------------------------------------
Total                        6   100.00%
----------------------------------------


In [10]:
summary, report_paths = generate_report(classification_results, OUTPUT_DIR, base_name=base_name)
report_paths

         PRODEXA ANALYSIS
Image: demo_basket
Generated: 2026-09-17T17:07:58

Total Products Detected: 6

Category                 Count         %
----------------------------------------
bottle-like                  2    33.33%
canister-like                2    33.33%
bag-like                     1    16.67%
box-like                     1    16.67%
----------------------------------------
Total                        6   100.00%
----------------------------------------


{'json': '../outputs\\reports\\demo_basket_report.json',
 'csv': '../outputs\\reports\\demo_basket_report.csv',
 'txt': '../outputs\\reports\\demo_basket_report.txt'}

In [11]:
with open(report_paths["json"]) as f:
    reloaded = json.load(f)
reloaded["total_products"], reloaded["category_counts"]

(6, {'bottle-like': 2, 'canister-like': 2, 'bag-like': 1, 'box-like': 1})

In [12]:
chart_paths = generate_charts(summary, OUTPUT_DIR, base_name=base_name)
chart_paths

{'bar': '../outputs\\charts\\demo_basket_bar_chart.png',
 'pie': '../outputs\\charts\\demo_basket_pie_chart.png'}

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread(chart_paths["bar"]))
axes[0].axis("off")
axes[0].set_title("Bar chart (counts)")

axes[1].imshow(plt.imread(chart_paths["pie"]))
axes[1].axis("off")
axes[1].set_title("Pie chart (distribution)")
plt.tight_layout()
plt.show()

C:\Users\dewmi\AppData\Local\Temp\ipykernel_23704\3653978892.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
if image is not None and detections is not None:
    labels_by_id = {r["det_id"]: f"{r['label']} {r['confidence']*100:.0f}%" for r in classification_results}
    annotated = draw_detections(image, detections, labels=labels_by_id)

    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"Annotated: {base_name}")
    plt.show()
else:
    print("No real image/detections available in this run (demo data only) - nothing to annotate.")

No real image/detections available in this run (demo data only) - nothing to annotate.


In [16]:
demo_image_2 = [
    {"det_id": 1, "label": "bottle-like",   "confidence": 0.91},
    {"det_id": 2, "label": "box-like",      "confidence": 0.85},
    {"det_id": 3, "label": "box-like",      "confidence": 0.72},
]
demo_image_3 = [
    {"det_id": 1, "label": "bag-like",      "confidence": 0.80},
    {"det_id": 2, "label": "canister-like", "confidence": 0.69},
]

summary_1 = summary  
summary_2 = build_summary(demo_image_2, image_name="demo_basket_2")
summary_3 = build_summary(demo_image_3, image_name="demo_basket_3")

batch_summary = aggregate_summaries([summary_1, summary_2, summary_3])
print_summary(batch_summary)

         PRODEXA ANALYSIS
Image: batch of 3 images
Generated: 2026-09-17T17:08:57

Total Products Detected: 11

Category                 Count         %
----------------------------------------
bottle-like                  3    27.27%
box-like                     3    27.27%
canister-like                3    27.27%
bag-like                     2    18.18%
----------------------------------------
Total                       11   100.00%
----------------------------------------


In [17]:
reports_dir = os.path.join(OUTPUT_DIR, "reports")
save_summary_json(batch_summary, os.path.join(reports_dir, "batch_report.json"))
save_summary_csv(batch_summary, os.path.join(reports_dir, "batch_report.csv"))
save_summary_text(batch_summary, os.path.join(reports_dir, "batch_report.txt"))

batch_chart_paths = generate_charts(batch_summary, OUTPUT_DIR, base_name="batch")
batch_chart_paths

{'bar': '../outputs\\charts\\batch_bar_chart.png',
 'pie': '../outputs\\charts\\batch_pie_chart.png'}

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(plt.imread(batch_chart_paths["bar"]))
axes[0].axis("off")
axes[0].set_title("Batch bar chart (counts)")

axes[1].imshow(plt.imread(batch_chart_paths["pie"]))
axes[1].axis("off")
axes[1].set_title("Batch pie chart (distribution)")
plt.tight_layout()
plt.show()

C:\Users\dewmi\AppData\Local\Temp\ipykernel_23704\337465369.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
for sub in ["reports", "charts"]:
    d = os.path.join(OUTPUT_DIR, sub)
    print(f"{d}/")
    for f in sorted(os.listdir(d)):
        print(f"   {f}")

../outputs\reports/
   .gitkeep.txt
   batch_report.csv
   batch_report.json
   batch_report.txt
   demo_basket_report.csv
   demo_basket_report.json
   demo_basket_report.txt
../outputs\charts/
   .gitkeep.txt
   batch_bar_chart.png
   batch_pie_chart.png
   demo_basket_bar_chart.png
   demo_basket_pie_chart.png
